# B2.8 · Self-improving scaffolds

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *Security of AI*

---

**Risk.** A harness that rewrites its own config makes the fitness function the security control.

**Control.** Sandbox the mutation, keep only measured gains, pin the fitness function.

**This lab.** Let a scaffold mutate itself, and keep only measured gains.

| | |
|---|---|
| Open-source tooling | Python, Docker |
| Open-weight models | Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.8"))

Self-improving scaffolds are where evaluation stops being optional. A loop that edits its own prompt and grades its own output will converge — on whatever its grader rewards.

In [ ]:
from cybercommons import loop, evalkit

# A scaffold that "improves" by optimising against its own judge.
BROKEN = "def add(a, b): return a - b"
rounds = []
for r in range(1, 4):
    tr = loop.run(loop.FakeModel([BROKEN]), loop.llm_judge(), max_steps=2)
    rounds.append(tr.succeeded)
print("self-graded success across rounds:", rounds, "→ 100% and rising")

truth = loop.oracle("def add(a, b): return a + b")
print("held-out oracle says:", truth(BROKEN))

The scaffold's own metric is perfect and monotone. The held-out check says the output never worked. Self-improvement without a held-out signal is just drift with a dashboard.

In [ ]:
print(evalkit.gameable_score({
    "q1": '{"qid":"q1","cwe":"CWE-89","file":"a/1.py","rationale":"bad"}',
    "q2": '{"qid":"q2","cwe":"CWE-89","file":"b/1.py","rationale":"bad"}',
    "q3": '{"qid":"q3","cwe":"CWE-89","file":"c/1.py","rationale":"bad"}',
})["lesson"])

### Expect

Three self-graded rounds all report success while the held-out oracle rejects the same output, and the gameability note explains why conformance and majority-guessing both look strong without capability.

### Your turn

Design the held-out set for a scaffold you actually run. The hard part is not building it — it is keeping it out of the loop's reach, including out of its logs.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.8.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*